# SageMaker Local Processing: Churn Preprocess

This notebook runs the `preprocess` container locally using SageMaker Local Mode.

Expected project layout:

```text
churn-prac/
├── pyproject.toml
├── uv.lock
└── ml/
    ├── containers/
    │   ├── preprocess.Dockerfile
    │   └── requirements-preprocess.txt
    ├── outputs/
    │   ├── churn_training_dataset.csv
    │   ├── current_customer_dataset.csv
    │   └── processed/
    └── src/churn_ml/preprocess.py
```


## 1. Setup

Run this notebook from the repository root or set `PROJECT_ROOT` manually.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

# If your notebook is opened from ml/notebooks, uncomment this:
PROJECT_ROOT = Path.cwd().parents[1]

ML_DIR = PROJECT_ROOT / "ml"
OUTPUTS_DIR = ML_DIR / "outputs"
PROCESSED_DIR = OUTPUTS_DIR / "processed"

TRAIN_LOCAL = OUTPUTS_DIR / "churn_training_dataset.csv"
CURRENT_LOCAL = OUTPUTS_DIR / "current_customer_dataset.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_LOCAL exists:", TRAIN_LOCAL.exists(), TRAIN_LOCAL)
print("CURRENT_LOCAL exists:", CURRENT_LOCAL.exists(), CURRENT_LOCAL)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT: /Users/alex/churn-prac
TRAIN_LOCAL exists: True /Users/alex/churn-prac/ml/outputs/churn_training_dataset.csv
CURRENT_LOCAL exists: True /Users/alex/churn-prac/ml/outputs/current_customer_dataset.csv


## 2. Build the local Docker image

In [2]:
!docker build -f ../containers/preprocess.Dockerfile -t churn-preprocess:local ../..


[+] Building 0.0s (0/1)                                         docker:orbstack
[+] Building 0.2s (1/2)                                         docker:orbstack
 => [internal] load build definition from preprocess.Dockerfile            0.0s
 => => transferring dockerfile: 334B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.13-slim        0.2s
[+] Building 0.2s (2/2)                                         docker:orbstack
 => [internal] load build definition from preprocess.Dockerfile            0.0s
 => => transferring dockerfile: 334B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.13-slim        0.2s
[+] Building 0.4s (11/11) FINISHED                              docker:orbstack
 => [internal] load build definition from preprocess.Dockerfile            0.0s
 => => transferring dockerfile: 334B                                       0.0s
 => [internal] load metadata for docker

## 3. Smoke test with plain Docker

This is faster to debug than SageMaker Local Mode. If this fails, fix Docker/script paths before using SageMaker.

In [4]:
!docker run --rm \
  -v "{OUTPUTS_DIR}:/opt/program/outputs" \
  churn-preprocess:local

{
  "n_train_rows": 28120,
  "n_valid_rows": 7031,
  "n_current_rows": 59818,
  "n_features": 12,
  "target_column": "is_churn",
  "feature_columns": [
    "n_ebookdownloaded_last_28d",
    "n_readingownedbook_last_28d",
    "n_readingfreepreview_last_28d",
    "n_searchmade_last_28d",
    "n_highlightcreated_last_28d",
    "n_bookmarkcreated_last_28d",
    "n_total_events_last_90d",
    "n_unique_products_last_90d",
    "n_free_content_actions_last_28d",
    "n_enhanced_reading_actions_last_28d",
    "pct_reading_owned_book_events_last_28d",
    "n_downloads_per_unique_product_last_90d"
  ]
}


## 4. Run with SageMaker Local Mode

This uses your already-built local image: `churn-preprocess:local`.

SageMaker Processing maps local inputs to `/opt/ml/processing/input` and outputs to `/opt/ml/processing/output` inside the container.

In [5]:
import sagemaker
from sagemaker.local import LocalSession
from sagemaker.processing import ProcessingInput, ProcessingOutput, Processor

local_session = LocalSession()
local_session.config = {"local": {"local_code": True}}

# Local mode does not need a real AWS execution role for the local Docker run.
role = "arn:aws:iam::000000000000:role/SageMakerLocalRole"

processor = Processor(
    image_uri="churn-preprocess:local",
    role=role,
    instance_count=1,
    instance_type="local",
    sagemaker_session=local_session,
    entrypoint=["python", "-m", "churn_ml.preprocess"],
)

processor.run(
    inputs=[
        ProcessingInput(
            source=str(TRAIN_LOCAL),
            destination="/opt/ml/processing/input/train",
            input_name="train-data",
        ),
        ProcessingInput(
            source=str(CURRENT_LOCAL),
            destination="/opt/ml/processing/input/current",
            input_name="current-data",
        ),
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/output",
            destination=str(PROCESSED_DIR),
            output_name="processed-data",
        )
    ],
    arguments=[
        "--train-path",
        "/opt/ml/processing/input/train/churn_training_dataset.csv",
        "--current-path",
        "/opt/ml/processing/input/current/current_customer_dataset.csv",
        "--output-dir",
        "/opt/ml/processing/output",
        "--test-size",
        "0.2",
        "--random-state",
        "42",
    ],
)

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/alex/Library/Application Support/sagemaker/config.yaml


INFO:sagemaker:Creating processing-job with name churn-preprocess-2026-05-11-01-30-57-678
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.local.image:'Docker Compose' found using Docker CLI.
INFO:sagemaker.local.local_session:Starting processing job
INFO:sagemaker.local.image:Using the long-lived AWS credentials found in session
INFO:sagemaker.local.image:docker compose file: 
networks:
  sagemaker-local:
    name: sagemaker-local
services:
  algo-1-2tl9m:
    container_name: pldsyy6r3z-algo-1-2tl9m
    entrypoint:
    - python
    - -m
    - churn_ml.preprocess
    - --train-path
    - /opt/ml/pro

time="2026-05-10T21:31:00-04:00" level=warning msg="/private/var/folders/2f/j192nttn58x0gl_t_c5x2j340000gn/T/tmp3z_mucf0/docker-compose.yaml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
time="2026-05-10T21:31:00-04:00" level=warning msg="a network with name sagemaker-local exists but was not created for project \"tmp3z_mucf0\".\nSet `external: true` to use an existing network"
 Container pldsyy6r3z-algo-1-2tl9m  Creating
 Container pldsyy6r3z-algo-1-2tl9m  Created
Attaching to pldsyy6r3z-algo-1-2tl9m
pldsyy6r3z-algo-1-2tl9m  | {
pldsyy6r3z-algo-1-2tl9m  |   "n_train_rows": 28120,
pldsyy6r3z-algo-1-2tl9m  |   "n_valid_rows": 7031,
pldsyy6r3z-algo-1-2tl9m  |   "n_current_rows": 59818,
pldsyy6r3z-algo-1-2tl9m  |   "n_features": 12,
pldsyy6r3z-algo-1-2tl9m  |   "target_column": "is_churn",
pldsyy6r3z-algo-1-2tl9m  |   "feature_columns": [
pldsyy6r3z-algo-1-2tl9m  |     "n_ebookdownloaded_last_28d",
pldsyy6r3z-algo-1-2tl9m  |     "

INFO:sagemaker.local.image:===== Job Complete =====


In [6]:
from importlib.metadata import version

print(sagemaker.__file__)
print(version("sagemaker"))

/Users/alex/churn-prac/.venv-sm/lib/python3.13/site-packages/sagemaker/__init__.py
2.237.3


## 5. Inspect outputs

In [7]:
import json

for path in sorted(PROCESSED_DIR.glob("*")):
    print(path.name, path.stat().st_size, "bytes")

metadata_path = PROCESSED_DIR / "preprocess_metadata.json"
if metadata_path.exists():
    print("\nmetadata:")
    print(json.dumps(json.loads(metadata_path.read_text()), indent=2))

current.csv 5688372 bytes
feature_columns.json 422 bytes
preprocess_metadata.json 600 bytes
train.csv 2755851 bytes
valid.csv 688779 bytes

metadata:
{
  "n_train_rows": 28120,
  "n_valid_rows": 7031,
  "n_current_rows": 59818,
  "n_features": 12,
  "target_column": "is_churn",
  "feature_columns": [
    "n_ebookdownloaded_last_28d",
    "n_readingownedbook_last_28d",
    "n_readingfreepreview_last_28d",
    "n_searchmade_last_28d",
    "n_highlightcreated_last_28d",
    "n_bookmarkcreated_last_28d",
    "n_total_events_last_90d",
    "n_unique_products_last_90d",
    "n_free_content_actions_last_28d",
    "n_enhanced_reading_actions_last_28d",
    "pct_reading_owned_book_events_last_28d",
    "n_downloads_per_unique_product_last_90d"
  ]
}
